In [41]:
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import numpy as np
from sklearn.impute import SimpleImputer
# === 1. Drop constant columns ===
class ConstantColumnRemover(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.cols_to_keep = [col for col in X.columns if X[col].nunique() > 1]
        return self
    def transform(self, X):
        return X[self.cols_to_keep]
    
# === 2. NAFiller ===
class NAFiller:
    def __init__(self, num_strategy='constant', num_fill_value=999,
                 cat_strategy='constant', cat_fill_value='Missing'):
        self.num_strategy = num_strategy
        self.num_fill_value = num_fill_value
        self.cat_strategy = cat_strategy
        self.cat_fill_value = cat_fill_value
        self.num_imputer = None
        self.cat_imputer = None
        self.num_cols = []
        self.cat_cols = []

    def fit(self, X, y=None):
        # Identify numerical and categorical columns
        self.num_cols = X.select_dtypes(include=['number']).columns.tolist()
        self.cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        
        # Fit numerical imputer
        self.num_imputer = SimpleImputer(strategy=self.num_strategy, fill_value=self.num_fill_value)
        if self.num_cols:
            self.num_imputer.fit(X[self.num_cols])
        
        # Fit categorical imputer
        self.cat_imputer = SimpleImputer(strategy=self.cat_strategy, fill_value=self.cat_fill_value)
        if self.cat_cols:
            self.cat_imputer.fit(X[self.cat_cols])
        
        print("finished fitting NAFiller")

        return self

    def transform(self, X):
        X = X.copy()
        # Transform numerical columns
        if self.num_cols:
            X[self.num_cols] = self.num_imputer.transform(X[self.num_cols])
        # Transform categorical columns
        if self.cat_cols:
            X[self.cat_cols] = self.cat_imputer.transform(X[self.cat_cols])
        print("finished transforming NAFiller")
        return X

# === 3. Encoder ===

class Encoder(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=3):
        self.threshold = threshold
        self.ohe_cols = []
        self.woe_cols = []
        self.woe_maps = {}
        self.ohe_df_columns = []

    def fit(self, X, y):
        X = X.copy()
        y = pd.Series(y).reset_index(drop=True)
        
        for col in X.select_dtypes(include=['object', 'category']):
            unique_vals = X[col].nunique(dropna=True)
            if unique_vals < self.threshold:
                self.ohe_cols.append(col)
            else:
                self.woe_cols.append(col)
                self.woe_maps[col] = self._compute_woe(X[col], y)

        if self.ohe_cols:
            ohe_df = pd.get_dummies(X[self.ohe_cols], dummy_na=True)
            self.ohe_df_columns = ohe_df.columns.tolist()
        print("finished fitting Encoder")
        
        return self

    def transform(self, X):
        X = X.copy()
        output = X.drop(columns=self.ohe_cols + self.woe_cols, errors='ignore')

        # Apply WOE encoding
        for col in self.woe_cols:
            woe_map = self.woe_maps[col]
            X[col] = X[col].map(woe_map).fillna(0)  # fill unknown categories with 0
            output[col] = X[col]

        # Apply One-Hot Encoding
        if self.ohe_cols:
            ohe_df = pd.get_dummies(X[self.ohe_cols], dummy_na=True)
            # Align with training columns to ensure consistency
            ohe_df = ohe_df.reindex(columns=self.ohe_df_columns, fill_value=0)
            output = pd.concat([output, ohe_df], axis=1)
        print("finished transforming Encoder")
        return output

    def _compute_woe(self, feature_col, target_col):
        df = pd.DataFrame({'feature': feature_col, 'target': target_col})
        df = df.dropna(subset=['feature'])  # Drop missing for WOE mapping
        grouped = df.groupby('feature')

        # total good and bad
        total_good = (df['target'] == 0).sum()
        total_bad = (df['target'] == 1).sum()

        woe_map = {}
        for val, group in grouped:
            good = (group['target'] == 0).sum()
            bad = (group['target'] == 1).sum()

            # Avoid division by zero
            epsilon = 1e-6
            good_ratio = good / total_good if total_good else epsilon
            bad_ratio = bad / total_bad if total_bad else epsilon
            woe = np.log((good_ratio + epsilon) / (bad_ratio + epsilon))

            woe_map[val] = woe

        return woe_map



# === 4. Feature Engineering ===
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, origin_date=pd.Timestamp('2017-11-30')):
        self.origin_date = origin_date

    def fit(self, X, y=None):
        print("finished fitting FeatureEngineer")
        return self

    def transform(self, X):
        X = X.copy()
        if 'TransactionDT' in X.columns:
            X['datetime'] = self.origin_date + pd.to_timedelta(X['TransactionDT'], unit='s')
            X['weekday'] = X['datetime'].dt.weekday
            X['month'] = X['datetime'].dt.month
            X.drop(columns=['datetime'], inplace=True)

        if 'TransactionAmt' in X.columns and 'card1' in X.columns:
            X['card1_avg_amt'] = X.groupby('card1')['TransactionAmt'].transform('mean')
        print('finished transforming FeatureEngineer')
        return X

# === 5. Correlation Filter ===
class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.98):
        self.threshold = threshold
        self.cols_to_keep = []
        self.corr_matrix = None

    def fit(self, X, y=None):
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        to_drop = [column for column in upper.columns if any(upper[column] > self.threshold)]
        self.cols_to_keep = [col for col in X.columns if col not in to_drop]
        self.corr_matrix = corr

        # Save outputs
        self.corr_matrix.to_csv('saved_corr_matrix.csv')
        pd.Series(self.cols_to_keep).to_csv('kept_columns.csv', index=False)
        print("finished fitting CorrelationFilter")
        return self

    def transform(self, X):
        print("finished transforming CorrelationFilter")
        return X[self.cols_to_keep]

# === 6. Ensure train/test match ===
class ColumnAligner(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.columns = None

    def fit(self, X, y=None):
        self.columns = X.columns
        print("finished fitting ColumnAligner")
        return self

    def transform(self, X):
        for col in self.columns:
            if col not in X:
                X[col] = 0
        print("finished transforming ColumnAligner")
        return X[self.columns]
    
# === 6. scaler ===

class Scaler(BaseEstimator, TransformerMixin):
    def __init__(self, scaler=None):
        self.scaler = scaler if scaler is not None else StandardScaler()
        self.columns = []

    def fit(self, X, y=None):
        X = X.copy()
        self.columns = X.select_dtypes(include=[np.number]).columns.tolist()
        self.scaler.fit(X[self.columns])
        print("finished fitting Scaler")
        return self

    def transform(self, X):
        X = X.copy()
        X[self.columns] = self.scaler.transform(X[self.columns])
        print("finished transforming Scaler")
        return X


# load data

In [2]:
import pandas as pd
train_identity  = pd.read_csv('ieee-fraud-detection/train_identity.csv')
train_transaction  = pd.read_csv('ieee-fraud-detection/train_transaction.csv')
test_identity = pd.read_csv('ieee-fraud-detection/test_identity.csv')
test_transaction = pd.read_csv('ieee-fraud-detection/test_transaction.csv')
# Normalize column names (replace '-' with '_')
train_identity.columns = train_identity.columns.str.replace('-', '_')
test_identity.columns = test_identity.columns.str.replace('-', '_')
train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')
test = pd.merge(test_transaction, test_identity, on='TransactionID', how='left')

In [42]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from sklearn.feature_selection import RFE

full_pipeline = ImbPipeline(steps=[
    ('na_filler', NAFiller()),
    ('remove_constants', ConstantColumnRemover()),
    ('feature_engineering', FeatureEngineer()),
    ('categorical_encoding', Encoder(threshold=4)),
    ('correlation_filter', CorrelationFilter(threshold=0.80)),
    ('align_columns', ColumnAligner()),
    ('undersampler', RandomUnderSampler(sampling_strategy=0.1, random_state=42)),
    ('rfe', RFE(estimator=XGBClassifier(eval_metric='logloss'))),  # n_features to be tuned
    ('classifier', XGBClassifier(eval_metric='logloss'))
])


In [ ]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical

search_space = {
    # Categorical encoder
    'categorical_encoding__threshold': Integer(2, 10),

    # NA filling
    'na_filler__num_strategy': Categorical(['zero', '999', 'mean']),

    # Correlation filter
    'correlation_filter__threshold': Real(0.85, 0.99),

    # Undersampling
    'undersampler__sampling_strategy': Real(0.05, 0.5),

    # RFE
    'rfe__n_features_to_select': Integer(10, 50),

    # XGBoost hyperparams
    'classifier__max_depth': Integer(3, 10),
    'classifier__learning_rate': Real(0.01, 0.3, prior='log-uniform'),
    'classifier__n_estimators': Integer(100, 500),
    'classifier__subsample': Real(0.5, 1.0),
    'classifier__colsample_bytree': Real(0.5, 1.0),
}


In [ ]:
search = BayesSearchCV(
    estimator=pipeline,
    search_spaces=search_space,
    cv=3,
    n_iter=50,  # increase for better accuracy
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2,
    random_state=42
)

search.fit(X, y)
print(search.best_params_)
print(search.best_score_)


In [43]:

# Usage
X = train.drop(columns=['isFraud', 'TransactionID'], errors='ignore')
y = train['isFraud']
X_test = test.drop(columns=['TransactionID'], errors='ignore')

In [45]:
full_pipeline.fit(X,y)


finished fitting NAFiller
finished transforming NAFiller
finished fitting FeatureEngineer
finished transforming FeatureEngineer
finished fitting Encoder
finished transforming Encoder
finished fitting CorrelationFilter
finished transforming CorrelationFilter
finished fitting ColumnAligner
finished transforming ColumnAligner


C:\Users\alex\Documents\university_work\ML\ML-homework_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [23:40:31] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Pipeline(steps=[('na_filler', <__main__.NAFiller object at 0x000001C34959EA50>),
                ('remove_constants', ConstantColumnRemover()),
                ('feature_engineering', FeatureEngineer()),
                ('categorical_encoding', Encoder(threshold=4)),
                ('correlation_filter', CorrelationFilter(threshold=0.8)),
                ('align_columns', ColumnAligner()),
                ('undersampler',
                 RandomUnderSampler(random_state=42,...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=None, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=None, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [48]:
y_pred = full_pipeline.predict_proba(X_test)


finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner


In [49]:
display(y_pred)

array([[0.9942919 , 0.0057081 ],
       [0.99063766, 0.00936232],
       [0.9897776 , 0.0102224 ],
       ...,
       [0.9799867 , 0.02001326],
       [0.98075753, 0.01924249],
       [0.9671518 , 0.03284818]], shape=(506691, 2), dtype=float32)

In [50]:
from sklearn.model_selection import cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = cross_validate(
    full_pipeline, X, y,
    cv=cv,
    scoring=['roc_auc', 'accuracy', 'f1'],
    return_train_score=True
)

print("Average ROC AUC:", results['test_roc_auc'].mean())


finished fitting NAFiller
finished transforming NAFiller
finished fitting FeatureEngineer
finished transforming FeatureEngineer
finished fitting Encoder
finished transforming Encoder
finished fitting CorrelationFilter
finished transforming CorrelationFilter
finished fitting ColumnAligner
finished transforming ColumnAligner


C:\Users\alex\Documents\university_work\ML\ML-homework_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [00:00:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished fitting NAFiller
finished transforming NAFiller
finished fitting FeatureEngineer
finished transforming FeatureEngineer
finished fitting Encoder
finished transforming Encoder
finished fitting CorrelationFilter
finished transforming CorrelationFilter
finished fitting ColumnAligner
finished tr

C:\Users\alex\Documents\university_work\ML\ML-homework_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [00:05:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished fitting NAFiller
finished transforming NAFiller
finished fitting FeatureEngineer
finished transforming FeatureEngineer
finished fitting Encoder
finished transforming Encoder
finished fitting CorrelationFilter
finished transforming CorrelationFilter
finished fitting ColumnAligner
finished tr

...

finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished fitting NAFiller
finished transforming NAFiller
finished fitting FeatureEngineer
finished transforming FeatureEngineer
finished fitting Encoder
finished transforming Encoder
finished fitting CorrelationFilter
finished transforming CorrelationFilter
finished fitting ColumnAligner
finished tr

C:\Users\alex\Documents\university_work\ML\ML-homework_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [00:15:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished fitting NAFiller
finished transforming NAFiller
finished fitting FeatureEngineer
finished transforming FeatureEngineer
finished fitting Encoder
finished transforming Encoder
finished fitting CorrelationFilter
finished transforming CorrelationFilter
finished fitting ColumnAligner
finished tr

C:\Users\alex\Documents\university_work\ML\ML-homework_2\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [00:20:41] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
finished transforming NAFiller
finished transforming FeatureEngineer
finished transforming Encoder
finished transforming CorrelationFilter
finished transforming ColumnAligner
Average ROC AUC: 0.9300109232952456


In [52]:
print("Average f1:", results['test_f1'].mean())
print("Average f1:", results['train_f1'].mean())


Average f1: 0.5984431202505542
Average f1: 0.6773546251769098


In [ ]:
# from xgboost import XGBClassifier
# from sklearn.model_selection import StratifiedKFold
# from skopt import BayesSearchCV
# from imblearn.pipeline import Pipeline as ImbPipeline
# from imblearn.under_sampling import RandomUnderSampler
# from sklearn.feature_selection import RFE
# 
# full_pipeline = ImbPipeline(steps=[
#     ('na_filler', NAFiller()),
#     ('remove_constants', ConstantColumnRemover()),
#     ('feature_engineering', FeatureEngineer()),
#     ('categorical_encoding', Encoder(threshold=4)),
#     ('correlation_filter', CorrelationFilter(threshold=0.80)),
#     ('align_columns', ColumnAligner()),
#     ('undersampler', RandomUnderSampler(sampling_strategy=0.1, random_state=42)),
#     ('rfe', RFE(estimator=XGBClassifier(eval_metric='logloss'))),  # n_features to be tuned
#     ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss'))
# ])

# debugging methods, do not run this otherwise

In [21]:
transformed_X = X.copy()
transformed_y = y.copy()
transformed_X_test = X_test.copy()

In [22]:
na_filler = NAFiller()
na_filler.fit(X,y)
print(na_filler.num_imputer)
print(na_filler.cat_imputer)

SimpleImputer(fill_value=999, strategy='constant')
SimpleImputer(fill_value='Missing', strategy='constant')


In [23]:
transformed_X = na_filler.transform(transformed_X)
transformed_X_test = na_filler.transform(transformed_X_test)
print(transformed_X.isnull().sum().sum())
print(transformed_X_test.isnull().sum().sum())

0
0


In [24]:
remove_constants = ConstantColumnRemover()
remove_constants.fit(transformed_X,y)

ConstantColumnRemover()

In [25]:
transformed_X = remove_constants.transform(transformed_X)
transformed_X_test = remove_constants.transform(transformed_X_test)

In [26]:

print(transformed_X)

        TransactionDT  TransactionAmt ProductCD    card1  card2  card3  \
0             86400.0           68.50         W  13926.0  999.0  150.0   
1             86401.0           29.00         W   2755.0  404.0  150.0   
2             86469.0           59.00         W   4663.0  490.0  150.0   
3             86499.0           50.00         W  18132.0  567.0  150.0   
4             86506.0           50.00         H   4497.0  514.0  150.0   
...               ...             ...       ...      ...    ...    ...   
590535     15811047.0           49.00         W   6550.0  999.0  150.0   
590536     15811049.0           39.50         W  10444.0  225.0  150.0   
590537     15811079.0           30.95         W  12037.0  595.0  150.0   
590538     15811088.0          117.00         W   7826.0  481.0  150.0   
590539     15811131.0          279.95         W  15066.0  170.0  150.0   

             card4  card5   card6  addr1  ...                id_31  id_32  \
0         discover  142.0  credit 

In [27]:
feature_engineering = FeatureEngineer()
feature_engineering.fit(transformed_X,y)


FeatureEngineer()

In [28]:
print(transformed_X_test.shape)
transformed_X = feature_engineering.transform(transformed_X)
transformed_X_test = feature_engineering.transform(transformed_X_test)
print(transformed_X_test.shape)


(506691, 432)
(506691, 435)


In [32]:
categorical_encoding = Encoder(threshold=4)
categorical_encoding.fit(transformed_X,transformed_y)

Encoder(threshold=4)

In [ ]:
transformed_X = categorical_encoding.transform(transformed_X)
transformed_X_test = categorical_encoding.transform(transformed_X_test)

In [36]:

num_cols = transformed_X_test.select_dtypes(include=['number','boolean']).columns
print(f"Number of numerical columns: {len(num_cols)}")
print(transformed_X_test.shape)

Number of numerical columns: 489
(506691, 489)


In [35]:
display(transformed_X_test)

,TransactionDT,TransactionAmt,card1,card2,card3,card5,addr1,addr2,dist1,dist2,...,id_37_T,id_37_nan,id_38_F,id_38_Missing,id_38_T,id_38_nan,DeviceType_Missing,DeviceType_desktop,DeviceType_mobile,DeviceType_nan
0,18403224.0,31.950,10409.0,111.0,150.0,226.0,170.0,87.0,1.0,999.0,...,False,False,False,True,False,False,True,False,False,False
1,18403263.0,49.000,4272.0,111.0,150.0,226.0,299.0,87.0,4.0,999.0,...,False,False,False,True,False,False,True,False,False,False
2,18403310.0,171.000,4476.0,574.0,150.0,226.0,472.0,87.0,2635.0,999.0,...,False,False,False,True,False,False,True,False,False,False
3,18403310.0,284.950,10989.0,360.0,150.0,166.0,205.0,87.0,17.0,999.0,...,False,False,False,True,False,False,True,False,False,False
4,18403317.0,67.950,18018.0,452.0,150.0,117.0,264.0,87.0,6.0,999.0,...,False,False,False,True,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
506686,34214279.0,94.679,13832.0,375.0,185.0,224.0,284.0,60.0,999.0,999.0,...,False,False,False,True,False,False,True,False,False,False
506687,34214287.0,12.173,3154.0,408.0,185.0,224.0,999.0,999.0,999.0,157.0,...,True,False,True,False,False,False,False,False,True,False
506688,34214326.0,49.000,16661.0,490.0,150.0,226.0,327.0,87.0,999.0,999.0,...,False,False,False,True,False,False,True,False,False,False
506689,34214337.0,202.000,16621.0,516.0,150.0,224.0,177.0,87.0,999.0,999.0,...,False,False,False,True,False,False,True,False,False,False
